# NISAR Mangrove Change Detection
## Notebook 01: Multi-Frequency SAR Archive Discovery & Audit (NASA ASF & ISRO Bhoonidhi)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/norp7183-ui/DANCING-WITH-SAR/blob/main/notebooks/01_asf_data_discovery.ipynb)

### 1. Objective & Study Context
- **Study Location**: Gosaba Community Development Block, Sundarbans Delta, West Bengal, India
- **Coordinates**: Latitude `21.9500° N`, Longitude `88.8500° E`
- **Investigation Window**: `2026-08-20` to `2026-10-01` (NISAR Mission Cycles 28 to 32)
- **Primary Goal**: Discover, audit, and cross-reference real satellite observations from **NISAR L-band (NASA ASF DAAC)** and **NISAR S-band (ISRO NRSC Bhoonidhi)**.

### 2. Scientific Working Rules & Physical Geometry
1. **Radar Look Geometry (Left-Looking)**:
   - NISAR operates in a nominal **Left-Looking** geometry relative to spacecraft velocity.
   - **Track 170 (Ascending Pass, heading ~350°)**: Illuminates towards the **West / North-West** at ~$39°$ mid-swath incidence at Gosaba.
   - **Track 019 (Descending Pass, heading ~190°)**: Illuminates towards the **East / South-East** at ~$43°$ mid-to-far swath incidence at Gosaba.
   - *Cross-track comparisons without incidence angle normalization are physically invalid.*
2. **Frequency Separation**:
   - **L-band GCOV**: Acquired via **NASA Alaska Satellite Facility (ASF) DAAC**.
   - **S-band GCOV**: Acquired via **ISRO NRSC Bhoonidhi** (operational release from **Cycle 25** onwards).
   - Equal polarization channel names ($HH$ in L-band vs $HH$ in S-band) represent fundamentally distinct physical penetration depths and interaction mechanisms.
3. **Environmental Covariates**:
   - Nighttime acquisitions are **not** automatically stable in the Sundarbans. Semi-diurnal tides ($4-6\text{ m}$) and monsoon rains occur day and night.
   - Acquisition UTC timestamps are metadata indices to query tide gauges (Canning / Sagar Island) and rainfall records (IMD AWS / ERA5).
4. **Decomposition Constraints**:
   - Quad-pol decompositions (Freeman-Durden, Cloude-Pottier) require complete $3 \times 3$ or $4 \times 4$ matrices and must **never** be applied to dual-pol data ($HH/HV$).

In [ ]:
# Step 1: Environment Setup & Library Import
import sys
import os
import json
import csv
from datetime import datetime
from pathlib import Path

# Auto-install dependencies if running in Google Colab
if 'google.colab' in sys.modules:
    print("[INFO] Google Colab environment detected. Installing dependencies...")
    !pip install -q asf_search geopandas shapely folium pandas
else:
    print("[INFO] Standard Python environment detected.")

import asf_search as asf
import pandas as pd
from shapely.geometry import shape, Polygon
import folium
from folium import plugins

print("[OK] Environment ready.")
print(f"asf_search version: {asf.__version__}")

In [ ]:
# Step 2: Secure NASA Earthdata Authentication (ASF DAAC Access)
def get_earthdata_session():
    """
    Authenticates with NASA Earthdata for downloading or restricted metadata.
    """
    username = None
    password = None

    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            username = userdata.get('EARTHDATA_USERNAME')
            password = userdata.get('EARTHDATA_PASSWORD')
            print("[INFO] Loaded credentials from Google Colab userdata.")
        except Exception:
            pass

    if not username or not password:
        import getpass
        print("Please provide NASA Earthdata credentials (https://urs.earthdata.nasa.gov):")
        username = input("Username: ").strip()
        password = getpass.getpass("Password: ").strip()

    try:
        session = asf.ASFSession()
        session.auth_with_creds(username=username, password=password)
        print("[SUCCESS] Authenticated with NASA Earthdata.")
        return session
    except Exception as e:
        print(f"[ERROR] Authentication failed: {e}")
        return None

# Uncomment to authenticate interactively:
# session = get_earthdata_session()

In [ ]:
# Step 3: Define Gosaba Study Area (AOI)
gosaba_bbox = {
    "name": "Gosaba Mangrove Study Area",
    "center_lat": 21.9500,
    "center_lon": 88.8500,
    "min_lon": 88.7000,
    "min_lat": 21.8000,
    "max_lon": 89.0000,
    "max_lat": 22.1000
}

aoi_wkt = (
    f"POLYGON(({gosaba_bbox['min_lon']} {gosaba_bbox['min_lat']}, "
    f"{gosaba_bbox['max_lon']} {gosaba_bbox['min_lat']}, "
    f"{gosaba_bbox['max_lon']} {gosaba_bbox['max_lat']}, "
    f"{gosaba_bbox['min_lon']} {gosaba_bbox['max_lat']}, "
    f"{gosaba_bbox['min_lon']} {gosaba_bbox['min_lat']}))"
)

print(f"Study Area: {gosaba_bbox['name']}")
print(f"Polygon WKT: {aoi_wkt}")

In [ ]:
# Step 4: Track A - Programmatic ASF Search for NISAR L-band GCOV
# Parameters verified against asf_search API:
# dataset='NISAR', processingLevel='GCOV', intersectsWith=aoi_wkt

search_params = {
    "dataset": "NISAR",
    "processingLevel": "GCOV",
    "intersectsWith": aoi_wkt,
    "start": "2026-08-20T00:00:00Z",
    "end": "2026-10-01T23:59:59Z"
}

print("[INFO] Searching NASA ASF DAAC Archive for NISAR L-band GCOV...")
try:
    results = asf.search(**search_params)
    print(f"[SUCCESS] Search returned {len(results)} granule(s).")
except Exception as e:
    print(f"[NOTICE] Direct search call encountered: {e}")
    # Load previously verified query cache if running in restricted offline environment
    cache_file = Path("../data/processed/asf_nisar_raw_response.json")
    if cache_file.exists():
        print(f"[INFO] Loading verified local archive response cache from {cache_file}...")
        raw_data = json.load(open(cache_file, encoding='utf-8'))
        results = raw_data[0] if isinstance(raw_data, list) and isinstance(raw_data[0], list) else raw_data
        print(f"[SUCCESS] Loaded {len(results)} verified granules from local cache.")
    else:
        results = []

In [ ]:
# Step 5: L-band Granule Metadata Auditing & Version Deduplication
# Load the verified L-band inventory
lband_csv = Path("../data/processed/asf_lband_inventory.csv")
if lband_csv.exists():
    df_lband = pd.read_csv(lband_csv)
    print(f"[OK] Loaded {len(df_lband)} audited L-band granules.")
    display(df_lband[[
        "calendar_date_utc", "start_time_utc", "relative_orbit_track",
        "flight_direction", "radar_look_direction", "polarization_category",
        "product_maturity", "product_counter", "version_selection_status"
    ]])
else:
    print("[WARNING] asf_lband_inventory.csv not found.")

### Step 6: Track B — ISRO Bhoonidhi S-Band GCOV Investigation

#### S-band API Specifications & Operational Status:
1. **Official API Search Endpoint**: `https://bhoonidhi-api.nrsc.gov.in/data/search`
   - *Direct Probe Result*: Returns `HTTP Error 401: Unauthorized`.
   - Programmatic access requires an active NRSC authorization header / Bearer Token (`bhoonidhi@nrsc.gov.in`).
2. **Operational Baseline**:
   - ISRO officially commenced Level-2 (L2-GCOV) S-SAR operational release with **Cycle 25** (July 8, 2026).
   - Cycles 28 to 32 fall within the operational release window.
3. **Verification Demarcation**:
   - **Planned Acquisitions (`PARTIALLY VERIFIED`)**: Scheduled under the NISAR Reference Observation Plan (ROP).
   - **Archived Granules over Gosaba (`NOT VERIFIED`)**: Exact spatial footprints and processing baselines cannot be claimed as confirmed without authenticated query results.
4. **Polarization Differences**:
   - S-band operates primarily in **Dual-Pol (HH/HV or VV/VH)** or **Circular-Linear Compact-Pol**, contrasting with L-band's full Quad-Pol mode on Track 019.

In [ ]:
# Step 7: Cross-Frequency Acquisition Comparison
comp_file = Path("../data/processed/cross_frequency_comparison.csv")
if comp_file.exists():
    df_comp = pd.read_csv(comp_file)
    print(f"[OK] Loaded {len(df_comp)} cross-frequency observation records.")
    display(df_comp[[
        "observation_date_utc", "exact_time_utc", "frequency", "data_provider",
        "track", "flight_direction", "radar_look_direction", "polarization_category",
        "verification_status"
    ]])
else:
    print("[WARNING] cross_frequency_comparison.csv not found.")

In [ ]:
# Step 8: Environmental Covariates Registration (UTC Timestamps)
# Rule: Never invent tide or rainfall measurements.
# Acquisition timestamps are recorded strictly as query indices for hydrographic and weather data.

print("=== MANDATORY ENVIRONMENTAL COVARIATE AUDIT ===")
print("Tidal Reference Stations: Canning (South 24 Parganas) & Sagar Island")
print("Meteorological Sources: IMD Canning/Alipore AWS & ERA5 Reanalysis")
print("\nTarget Overpass Timestamps for Environmental Gauge Linkage:")

unique_timestamps = sorted(list(set(df_lband["start_time_utc"])))
for ts in unique_timestamps:
    print(f"  - Overpass UTC: {ts} | Required: Tide Gauge Level (m) & Prior 24h/48h Rainfall (mm)")

In [ ]:
# Step 9: Interactive Map of Gosaba Study Area and Satellite Footprints
m = folium.Map(
    location=[gosaba_bbox['center_lat'], gosaba_bbox['center_lon']],
    zoom_start=11,
    tiles="CartoDB positron"
)

# Study Area Polygon
folium.Rectangle(
    bounds=[[gosaba_bbox['min_lat'], gosaba_bbox['min_lon']], [gosaba_bbox['max_lat'], gosaba_bbox['max_lon']]],
    color="#1b5e20",
    weight=2,
    fill=True,
    fill_color="#2e7d32",
    fill_opacity=0.15,
    popup="Gosaba Study Area (AOI)",
    tooltip="Gosaba AOI"
).add_to(m)

# Center Marker
folium.Marker(
    location=[gosaba_bbox['center_lat'], gosaba_bbox['center_lon']],
    icon=folium.Icon(color="green", icon="tree", prefix="fa"),
    popup="<b>Gosaba Center</b><br>Lat: 21.9500° N<br>Lon: 88.8500° E",
    tooltip="Gosaba Center"
).add_to(m)

folium.LayerControl().add_to(m)
print("[OK] Study Area Map rendered.")
m